# 6 · Detailed Stats  `[EVAL]`

The **full statistical tables** behind the figures in `0`-`1` — kept here so the analysis notebooks stay figure-led. Everything is full-conversation eval, paired by the 96 shared personas. Thin arms (<3 scored iters) are dropped to avoid NaN rows. For a reader who wants exact numbers.

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import exp3
from exp3 import stats, behavior, training, pref, figures, plots
S = exp3.notebook_setup()

arms on disk: [('PTO_LA0', 11), ('PTO_LA5', 6), ('GRPO_LA0', 6), ('GRPO_LA5', 2)]
scores_long: (16128, 19) | arms scored: ['GRPO_LA0', 'GRPO_LA5', 'PTO_LA0', 'PTO_LA5']
exports -> C:\Users\baruc\Desktop\Projects\Thesis_PTO_GRPO\Exp3_PTO_GRPO\eda\results


## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × rubric): final (and best) iteration vs base — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm), bootstrap 95% CI, trajectory ρ / slope.

In [2]:
MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
display(MR)
exp3.save_table(MR, "main_results_final", caption="Final iteration vs base, per arm x rubric, persona-paired (N=96): dz, Wilcoxon p (Holm), bootstrap CI, trajectory rho/slope. Thin arms dropped.")
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"), S.SCORES)
exp3.save_table(MRb, "main_results_best", caption="As main_results_final but vs each arm's BEST iteration (selection-sensitivity companion).")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,rubric,base,target_iter,target,delta,dz,effect,wilcoxon_p,ci_low,ci_high,traj_rho,traj_slope,p_holm
0,GRPO_LA0,Q1Q2,3.067,5,3.972,0.905,1.124,large,0.0,0.751,1.066,0.303,0.2104,0.0
1,GRPO_LA0,WAI-SR,2.890,5,3.208,0.319,0.492,small,0.0,0.194,0.448,0.147,0.0811,0.0
2,GRPO_LA0,CSQ-8,2.362,5,2.715,0.353,0.604,medium,0.0,0.240,0.471,0.134,0.0824,0.0
3,GRPO_LA0,MI-SAT,2.710,5,3.214,0.503,0.604,medium,0.0,0.332,0.679,0.147,0.1184,0.0
4,GRPO_LA0,MITI,3.224,5,3.940,0.716,0.979,large,0.0,0.573,0.862,0.314,0.1797,0.0
5,GRPO_LA0,Q1,3.021,5,3.869,0.848,1.048,large,0.0,0.696,1.008,0.264,0.2010,0.0
6,GRPO_LA0,Q2,3.113,5,4.075,0.962,1.097,large,0.0,0.794,1.140,0.337,0.2199,0.0
7,PTO_LA0,Q1Q2,3.000,10,4.260,1.259,1.429,large,0.0,1.086,1.434,0.353,0.1203,0.0
8,PTO_LA0,WAI-SR,2.845,10,3.497,0.653,0.968,large,0.0,0.525,0.786,0.226,0.0687,0.0
9,PTO_LA0,CSQ-8,2.384,10,2.945,0.561,0.805,large,0.0,0.419,0.701,0.171,0.0587,0.0


  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


'C:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables'

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm × rubric.

In [3]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
exp3.save_table(FR.round(4), "friedman_omnibus", caption="Friedman repeated-measures omnibus across iterations per arm x rubric (Kendall's W). Thin arms dropped.")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,metric,chi2,p,kendall_w,k_iters,n_personas
0,GRPO_LA0,Q1Q2,200.1633,0.0,0.4170,6,96
1,GRPO_LA0,WAI-SR,57.4288,0.0,0.1196,6,96
2,GRPO_LA0,CSQ-8,67.8660,0.0,0.1414,6,96
3,GRPO_LA0,MI-SAT,60.9824,0.0,0.1270,6,96
4,GRPO_LA0,MITI,170.6406,0.0,0.3555,6,96
5,GRPO_LA0,Q1,186.3109,0.0,0.3881,6,96
6,GRPO_LA0,Q2,181.7290,0.0,0.3786,6,96
7,PTO_LA0,Q1Q2,432.4498,0.0,0.4505,11,96
8,PTO_LA0,WAI-SR,254.6956,0.0,0.2653,11,96
9,PTO_LA0,CSQ-8,153.8643,0.0,0.1603,11,96


'C:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables'

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN base).

In [4]:
for arm in sorted(S.SCORES.arm.unique()):
    if arm in stats.thin_arms(S.SCORES): continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if PV.empty: continue
    print(f"=== {arm}: Q1+Q2 vs base (persona-paired) ==="); display(PV[["iteration","n","mean_delta","dz","p_holm","ci_low","ci_high"]].round(4))
    exp3.save_table(PV[["iteration","n","mean_delta","dz","p","p_holm","ci_low","ci_high"]].round(4), f"{arm}_Q1Q2_vs_base_paired", caption=f"{arm} each iteration vs base on Q1+Q2; persona-paired Wilcoxon, dz, Holm p, bootstrap CI.")

=== GRPO_LA0: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.2022,0.2129,0.0423,0.0202,0.3889
1,2,96,0.2926,0.3341,0.0013,0.1281,0.4740
2,3,96,0.9267,1.1494,0.0000,0.7694,1.0869
3,4,96,0.9376,1.1403,0.0000,0.7731,1.1017
4,5,96,0.9050,1.1238,0.0000,0.7506,1.0659


=== PTO_LA0: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.2632,0.2690,0.0352,0.0814,0.4545
1,2,96,0.4659,0.4587,0.0000,0.2651,0.6732
2,3,96,0.8145,0.8427,0.0000,0.6185,1.0041
3,4,96,1.0072,1.0782,0.0000,0.8208,1.1967
4,5,96,1.0141,1.0532,0.0000,0.8221,1.2057
5,6,96,1.1536,1.2035,0.0000,0.9693,1.3494
6,7,96,1.1287,1.1216,0.0000,0.9314,1.3231
7,8,96,1.2203,1.2504,0.0000,1.0300,1.4240
8,9,96,1.2381,1.3100,0.0000,1.0483,1.4242
9,10,96,1.2594,1.4288,0.0000,1.0859,1.4336


=== PTO_LA5: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.1775,0.1629,0.1293,-0.0378,0.3901
1,2,96,0.3048,0.2750,0.0369,0.0911,0.5242
2,3,96,0.6710,0.6714,0.0000,0.4645,0.8743
3,4,96,0.8847,0.8811,0.0000,0.6894,1.0835


## 4 · Cross-arm contrasts — PTO vs GRPO (matched K) & K0 vs K5  `[EVAL]`
**Purpose.** The method and look-ahead contrasts as paired tables at matched iterations (iteration-0 base rows dropped). Positive `mean_delta` = first term higher.

In [5]:
for K in sorted(S.SCORES.K.unique()):
    CMP = stats.paired_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"K={K}: no common PTO/GRPO iters"); continue
    view = CMP[["iteration","metric","n","mean_delta","dz","p_holm"]].round(4)
    print(f"=== PTO_LA{K} - GRPO_LA{K} (paired; + => PTO higher) ==="); display(view)
    exp3.save_table(view, f"PTO_vs_GRPO_LA{K}_paired", caption=f"PTO_LA{K} - GRPO_LA{K} at matched iterations; persona-paired Wilcoxon + dz + Holm.")
for method in ["PTO", "GRPO"]:
    CMP = stats.paired_k_comparison(S.SCORES, method)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"{method}: K0-vs-K5 not comparable yet"); continue
    view = CMP[["iteration","metric","mean_delta","dz","p_holm"]].round(4)
    print(f"=== {method}: LA0 - LA5 (+ => K0 higher) ==="); display(view)
    exp3.save_table(view, f"{method}_K0_vs_K5_paired", caption=f"{method} K0 - K5 at matched iterations; persona-paired Wilcoxon + dz + Holm.")

=== PTO_LA0 - GRPO_LA0 (paired; + => PTO higher) ===


,iteration,metric,n,mean_delta,dz,p_holm
7,1,Q1Q2,96,-0.0055,-0.0060,1.0000
8,1,WAI-SR,96,-0.0807,-0.1129,1.0000
9,1,CSQ-8,96,-0.0755,-0.1153,1.0000
10,1,MI-SAT,96,-0.1372,-0.1481,0.5110
11,1,MITI,96,0.0547,0.0580,1.0000
12,1,Q1,96,-0.0042,-0.0045,1.0000
13,1,Q2,96,-0.0067,-0.0070,1.0000
14,2,Q1Q2,96,0.1069,0.1127,1.0000
15,2,WAI-SR,96,0.0087,0.0111,1.0000
16,2,CSQ-8,96,0.0117,0.0154,1.0000


=== PTO_LA5 - GRPO_LA5 (paired; + => PTO higher) ===


,iteration,metric,n,mean_delta,dz,p_holm
7,1,Q1Q2,96,-0.0907,-0.0945,1.0
8,1,WAI-SR,96,-0.0391,-0.0550,1.0
9,1,CSQ-8,96,0.0156,0.0201,1.0
10,1,MI-SAT,96,0.0174,0.0170,1.0
11,1,MITI,96,-0.0885,-0.0941,1.0
12,1,Q1,96,-0.0687,-0.0687,1.0
13,1,Q2,96,-0.1127,-0.1145,1.0


=== PTO: LA0 - LA5 (+ => K0 higher) ===


,iteration,metric,mean_delta,dz,p_holm
7,1,Q1Q2,0.0828,0.0931,1.0000
8,1,WAI-SR,-0.0295,-0.0452,1.0000
9,1,CSQ-8,-0.0104,-0.0150,1.0000
10,1,MI-SAT,-0.0747,-0.0789,1.0000
11,1,MITI,0.1276,0.1514,0.7227
12,1,Q1,0.0687,0.0760,1.0000
13,1,Q2,0.0968,0.1032,1.0000
14,2,Q1Q2,0.1583,0.1830,0.4331
15,2,WAI-SR,-0.0148,-0.0239,1.0000
16,2,CSQ-8,0.0430,0.0644,1.0000


=== GRPO: LA0 - LA5 (+ => K0 higher) ===


,iteration,metric,mean_delta,dz,p_holm
7,1,Q1Q2,-0.0025,-0.0028,1.0
8,1,WAI-SR,0.0122,0.0178,1.0
9,1,CSQ-8,0.0807,0.1154,1.0
10,1,MI-SAT,0.0799,0.0867,1.0
11,1,MITI,-0.0156,-0.0165,1.0
12,1,Q1,0.0042,0.0044,1.0
13,1,Q2,-0.0092,-0.0102,1.0


## 5 · Climb rate, rankings, factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm (climb rate vs endpoint); the cross-model leaderboard; and the rubric PCA (PC1 share → are the 6 rubrics ~one factor?).

In [6]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, "Q1Q2") for a in sorted(S.SCORES.arm.unique())]), S.SCORES)
display(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4))
exp3.save_table(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4), "Q1Q2_slope_by_arm", caption="Q1+Q2 per-iteration OLS slope + Spearman rho per arm (climb rate). Thin arms dropped.")
PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in sorted(S.SCORES.arm.unique()) if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
exp3.save_table(PCA, "rubric_pca_pc1", caption="Variance explained by PC1 of the rubric scores per arm (dominant PC1 => rubrics ~ one latent factor).")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,metric,spearman_rho,ols_slope,peak_iter,final_iter
0,GRPO_LA0,Q1Q2,0.3034,0.2104,4.0,5.0
1,PTO_LA0,Q1Q2,0.3534,0.1203,10.0,10.0
2,PTO_LA5,Q1Q2,0.2720,0.2263,4.0,4.0


,arm,PC1_pct
0,GRPO_LA0,92.1
1,GRPO_LA5,93.3
2,PTO_LA0,90.7
3,PTO_LA5,91.6


'C:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables'